In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

repos = [
    'deepghs/fgo_voices_jp',
    'deepghs/azurlane_voices_jp',
    'deepghs/girlsfrontline_voices_jp',
]

for r in repos:
    snapshot_download(
        repo_id=r, 
        repo_type="dataset", local_dir=r.split('/')[1])

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:03<00:00,  1.66it/s]


In [3]:
files = glob('*_voices_jp/*.tar')
files

['girlsfrontline_voices_jp/voices.tar',
 'azurlane_voices_jp/voices.tar',
 'fgo_voices_jp/voices.tar']

In [4]:
def loop(files):
    files, _ = files
    import tarfile

    for f in tqdm(files):
        with tarfile.open(f, "r") as tar:
            tar.extractall(path=f.split('/')[0])
        os.remove(f)

In [5]:
multiprocessing(files, loop, cores = len(files), returned = False)

100%|██████████| 1/1 [00:04<00:00,  4.97s/it]


In [8]:
files = glob('*_voices_jp/table.parquet')
files = [f for f in files if 'arknights' not in f]
files

['girlsfrontline_voices_jp/table.parquet',
 'azurlane_voices_jp/table.parquet',
 'fgo_voices_jp/table.parquet']

In [11]:
rows = []
for f in files:
    print(f)
    df = pd.read_parquet(f)
    f_ = f.split('/')[0]
    for i in range(len(df)):
        t = df['voice_text'].iloc[i].strip()
        if len(t) < 2:
            continue
        id = df['id'].iloc[i]
        f = f'{f_}/{id}.ogg'
        if not os.path.exists(f):
            continue
        rows.append({
            'audio_filename': f,
            'text': t,
            'speaker': f"{f_}_{df['voice_actor_name'].iloc[i]}"
        })
len(rows)

girlsfrontline_voices_jp/table.parquet
azurlane_voices_jp/table.parquet
fgo_voices_jp/table.parquet


43492

In [12]:
rows[0]

{'audio_filename': 'girlsfrontline_voices_jp/char_1_M1873Mod_ALLHALLOWS_JP.ogg',
 'text': 'お化けとか怖くないからね！「…」のお礼だもん。でも、もうちょっと妖魔心をもっと方がいいと思う。',
 'speaker': 'girlsfrontline_voices_jp_Tanaka Aimi'}

In [13]:
import copy

def loop(rows):
    rows, _ = rows
    data = []
    for r in tqdm(rows):
        r = copy.copy(r)
        base = r['audio_filename'].split('/')[0] + '_audio'
        audio_filename = r['audio_filename'].replace('/', '-').replace('.ogg', '.mp3')
        os.makedirs(base, exist_ok=True)
        audio_filename = os.path.join(base, audio_filename)
        audio_np, sr = sf.read(r['audio_filename'])
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)
        r['audio_filename'] = audio_filename
        data.append(r)
    return data

In [14]:
data = loop((rows[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 13.67it/s]


In [16]:
data = multiprocessing(rows, loop, cores = 30)

100%|██████████| 1449/1449 [03:38<00:00,  6.64it/s]


In [17]:
audio_files = [d['audio_filename'] for d in data]

with open('voices_jp-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [18]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'girlsfrontline_voices_jp_audio/girlsfrontline_voices_jp-char_1_M1873Mod_ALLHALLOWS_JP.mp3',
 'text': 'お化けとか怖くないからね！「…」のお礼だもん。でも、もうちょっと妖魔心をもっと方がいいと思う。',
 'speaker': 'girlsfrontline_voices_jp_Tanaka Aimi'}

In [19]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'voices_jp')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 49.03ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  88%|████████▊ | 2.67MB / 3.02MB, 13.3MB/s  
Processing Files (1 / 1): 100%|██████████| 3.02MB / 3.02MB, 7.56MB/s  
Processing Files (1 / 1): 100%|██████████| 3.02MB / 3.02MB, 2.52MB/s  
New Data Upload: 100%|██████████| 3.02MB / 3.02MB, 2.52MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.54s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/7b3eaf04088f5bd5f5be6ac2e462d097207fa0e1', commit_message='Upload dataset', commit_description='', oid='7b3eaf04088f5bd5f5be6ac2e462d097207fa0e1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [21]:
for f in glob('*_voices_jp_audio'):
    print(f)
    if 'ark' in f:
        continue
    os.system(f'zip -rq {f}.zip {f}')

azurlane_voices_jp_audio
fgo_voices_jp_audio
arknights_voices_jp_audio
girlsfrontline_voices_jp_audio


In [22]:
for f in glob('*_voices_jp_audio.zip'):
    print(f)
    if 'ark' in f:
        continue
    os.system(f'hf upload malaysia-ai/Multilingual-TTS {f} --repo-type=dataset')

azurlane_voices_jp_audio.zip


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  azurlane_voices_jp_audio.zip:   2%|▏         | 37.8MB / 2.31GB            

Processing Files (0 / 1)      :   2%|▏         | 37.8MB / 2.31GB, 94.6MB/s  
New Data Upload               :  56%|█████▋    | 37.7MB / 67.0MB, 94.5MB/s  

Processing Files (0 / 1)      :   8%|▊         |  178MB / 2.31GB,  296MB/s  
New Data Upload               :  66%|██████▋   |  178MB /  268MB,  296MB/s  

Processing Files (0 / 1)      :  14%|█▍        |  322MB / 2.31GB,  403MB/s  
New Data Upload               :  80%|████████  |  322MB /  402MB,  403MB/s  

Processing Files (0 / 1)      :  20%|█▉        |  462MB / 2.31GB,  462MB/s  
New Data Upload               :  86%|████████▌ |  462MB /  536MB,  462MB/s  

Processing Files (0 / 1)      :  23%|██▎       |  533MB / 2.31GB,  444MB/s  
New Data Upload               :  88%|████████▊ |  533MB /  603MB,  444MB/s  



https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/blob/main/azurlane_voices_jp_audio.zip
arknights_voices_jp_audio.zip
fgo_voices_jp_audio.zip


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  fgo_voices_jp_audio.zip     :  99%|█████████▉| 51.5MB / 51.9MB            

Processing Files (0 / 1)      :  99%|█████████▉| 51.5MB / 51.9MB,  129MB/s  
New Data Upload               :  99%|█████████▉| 51.5MB / 51.9MB,  129MB/s  

  fgo_voices_jp_audio.zip     :  99%|█████████▉| 51.5MB / 51.9MB            

  fgo_voices_jp_audio.zip     :  99%|█████████▉| 51.5MB / 51.9MB            

  fgo_voices_jp_audio.zip     :  99%|█████████▉| 51.5MB / 51.9MB            

  fgo_voices_jp_audio.zip     :  99%|█████████▉| 51.5MB / 51.9MB            

Processing Files (1 / 1)      : 100%|██████████| 51.9MB / 51.9MB, 38.6MB/s  
New Data Upload               : 100%|██████████| 51.9MB / 51.9MB, 38.6MB/s  

Processing Files (1 / 1)      : 100%|██████████| 51.9MB / 51.9MB, 37.1MB/s  
New Data Upload               : 100%|██████████| 51.9MB / 51.9MB, 37.1MB/s  

https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/blob/main/fgo_voices_jp_audio.zip
girlsfrontline_voices_jp_audio.zip


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...tline_voices_jp_audio.zip:   8%|▊         | 53.5MB /  710MB            

Processing Files (0 / 1)      :   8%|▊         | 53.5MB /  710MB,  134MB/s  
New Data Upload               :  40%|███▉      | 53.5MB /  134MB,  134MB/s  

Processing Files (0 / 1)      :  28%|██▊       |  196MB /  710MB,  326MB/s  
New Data Upload               :  73%|███████▎  |  196MB /  268MB,  326MB/s  

Processing Files (0 / 1)      :  47%|████▋     |  332MB /  710MB,  415MB/s  
New Data Upload               :  82%|████████▏ |  331MB /  402MB,  414MB/s  

Processing Files (0 / 1)      :  67%|██████▋   |  477MB /  710MB,  477MB/s  
New Data Upload               :  89%|████████▉ |  477MB /  536MB,  476MB/s  

Processing Files (0 / 1)      :  75%|███████▌  |  533MB /  710MB,  444MB/s  
New Data Upload               :  88%|████████▊ |  533MB /  603MB,  444MB/s  



https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/blob/main/girlsfrontline_voices_jp_audio.zip
